In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from collections import Counter
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!apt-get install unrar

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
unrar is already the newest version (1:6.1.5-1ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 35 not upgraded.


In [ ]:
rar_path = "/content/drive/MyDrive/data_.rar"  # ← change ça selon l'emplacement exact
extract_dir = "/content/data"

!unrar x "{rar_path}" "{extract_dir}/"


UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/drive/MyDrive/data_.rar

Creating    /content/data                                             OK
Creating    /content/data/data_                                       OK
Creating    /content/data/data_/Buche                                 OK
Extracting  /content/data/data_/Buche/102.pts                              0%  OK 
Extracting  /content/data/data_/Buche/103.pts                              0%  OK 
Extracting  /content/data/data_/Buche/106.pts                              0%  OK 
Extracting  /content/data/data_/Buche/107.pts                              0%  OK 
Extracting  /content/data/data_/Buche/109.pts                              0%  OK 
Extracting  /content/data/data_/Buche/110.pts                              0%  OK 
Extracting  /content/data/data_/Buche/12.pts                               0%  OK 
Extracting 

In [ ]:
import numpy as np
import glob
import os

# Fonction pour charger les fichiers .xyz et .pls
def load_xyz_or_pls(path):
    try:
        return np.loadtxt(path, usecols=(0, 1, 2))
    except Exception as e:
        print(f"Erreur de chargement: {path}\n{e}")
        return np.zeros((0, 3))

# Fonction de sous-échantillonnage
def subsample(cloud, n=1024):
    if cloud.shape[0] >= n:
        idx = np.random.choice(cloud.shape[0], n, replace=False)
    else:
        idx = np.random.choice(cloud.shape[0], n, replace=True)
    return cloud[idx]

# Trouver tous les fichiers .xyz et .pls dans les sous-dossiers
files = glob.glob('/content/data/**/*', recursive=True)
files = [f for f in files if f.lower().endswith(('.xyz', '.pls','.txt'))]

X, y = [], []

for path in files:
    cloud = load_xyz_or_pls(path)
    if cloud.shape[1] != 3 or cloud.shape[0] < 10:
        continue  # Ignorer les fichiers vides ou incorrects
    X.append(subsample(cloud))
    label = os.path.basename(os.path.dirname(path))  # Nom du dossier = classe
    y.append(label)

X = np.array(X)
classes = sorted(set(y))
y = np.array([classes.index(label) for label in y])

print("✅ Chargement terminé :", X.shape, y.shape)


✅ Chargement terminé : (577, 1024, 3) (577,)


In [ ]:
save_path = "/content/pointcloud_dataset.npz"
np.savez_compressed(save_path, X=X, y=y, classes=classes)
print(f"✅ Fichier sauvegardé : {save_path}")

✅ Fichier sauvegardé : /content/pointcloud_dataset.npz


In [ ]:
data = np.load('/content/pointcloud_dataset.npz')
X = data['X']
y = data['y']
classes = data['classes']

In [ ]:
# Vectorisation : aplatir (1024, 3) → (3072,) - This is only used for sklearn models
X_flat = X.reshape(X.shape[0], -1)

# Division train/test - Use X_flat for sklearn models
X_train_flat, X_test_flat, y_train, y_test = train_test_split(X_flat, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# Modèles Sklearn (using flattened data)
models = {
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Neural Net": MLPClassifier(hidden_layer_sizes=(100,))
}

# Entraînement et évaluation Sklearn
for name, model in models.items():
    model.fit(X_train_flat, y_train) # Use X_train_flat
    y_pred = model.predict(X_test_flat) # Use X_test_flat
    acc = accuracy_score(y_test, y_pred)

    # Get unique labels present in y_test and filter target_names accordingly
    unique_test_labels = np.unique(y_test)
    filtered_target_names = [classes[i] for i in unique_test_labels]

    print(f"\n{name} Metrics:")
    print(f"Accuracy: {acc:.4f}")
    print("Classification Report:")
    # Pass unique_test_labels to the labels parameter
    print(classification_report(y_test, y_pred, labels=unique_test_labels, target_names=filtered_target_names))
    print("Confusion Matrix:")
    # The confusion matrix also benefits from knowing the actual labels present
    cm = confusion_matrix(y_test, y_pred, labels=unique_test_labels)
    print(cm)


SVM Metrics:
Accuracy: 0.5431
Classification Report:
              precision    recall  f1-score   support

       Buche       0.44      0.44      0.44        16
   Douglasie       0.72      0.49      0.58        37
       Eiche       0.00      0.00      0.00         4
       Esche       0.00      0.00      0.00         2
      Fichte       0.61      0.59      0.60        32
      Kiefer       0.00      0.00      0.00         5
    Roteiche       0.43      0.95      0.59        20

    accuracy                           0.54       116
   macro avg       0.31      0.35      0.32       116
weighted avg       0.53      0.54      0.51       116

Confusion Matrix:
[[ 7  0  0  0  0  0  9]
 [ 4 18  0  0 12  0  3]
 [ 1  1  0  0  0  0  2]
 [ 0  0  0  0  0  0  2]
 [ 1  4  0  0 19  0  8]
 [ 3  1  0  0  0  0  1]
 [ 0  1  0  0  0  0 19]]

KNN Metrics:
Accuracy: 0.4224
Classification Report:
              precision    recall  f1-score   support

       Buche       0.44      0.25      0.32        16

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/m


Decision Tree Metrics:
Accuracy: 0.4224
Classification Report:
              precision    recall  f1-score   support

       Buche       0.25      0.25      0.25        16
   Douglasie       0.56      0.54      0.55        37
       Eiche       0.29      0.50      0.36         4
       Esche       0.00      0.00      0.00         2
      Fichte       0.44      0.44      0.44        32
      Kiefer       0.00      0.00      0.00         5
    Roteiche       0.43      0.45      0.44        20

    accuracy                           0.42       116
   macro avg       0.28      0.31      0.29       116
weighted avg       0.42      0.42      0.42       116

Confusion Matrix:
[[ 4  3  2  0  1  1  5]
 [ 3 20  0  2 11  0  1]
 [ 1  0  2  0  0  0  1]
 [ 1  0  1  0  0  0  0]
 [ 1 10  2  0 14  0  5]
 [ 3  1  0  0  1  0  0]
 [ 3  2  0  0  5  1  9]]

Neural Net Metrics:
Accuracy: 0.4569
Classification Report:
              precision    recall  f1-score   support

       Buche       0.36      0.50   

In [ ]:
class PointCloudDataset(Dataset):
    def __init__(self, X, y):
        # Ensure X is a torch tensor with correct dtype
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
# 4. PointNet
class PointNet(nn.Module):
    def __init__(self, num_classes):
        super(PointNet, self).__init__()
        # PointNet expects (batch_size, num_points * num_dims) after flattening
        self.fc1 = nn.Linear(1024 * 3, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, num_classes)

    def forward(self, x):
        # Input x is (batch_size, num_points, num_dims) from the dataset
        x = x.view(-1, 1024 * 3)  # Flatten the point cloud to (batch_size, num_points * num_dims)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# num_classes should be based on the unique classes in the *full* dataset
num_classes = len(classes) # Use the full list of classes

pointnet = PointNet(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pointnet.parameters(), lr=0.001)

# --- Changes for PyTorch models start here ---
# Create datasets using the original 3D data (X)
# Need to perform the train/test split on the 3D data as well for consistency with PyTorch dataloaders
X_train, X_test, y_train_pt, y_test_pt = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Create PointCloudDataset instances for train and test sets using the 3D split data
train_dataset_pt = PointCloudDataset(X_train, y_train_pt)
test_dataset_pt = PointCloudDataset(X_test, y_test_pt)

# Create DataLoaders
train_loader_pt = DataLoader(train_dataset_pt, batch_size=32, shuffle=True)
test_loader_pt = DataLoader(test_dataset_pt, batch_size=32)
# --- Changes for PyTorch models end here ---


# Entraînement PointNet
def train_pointnet(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        # X_batch from loader is (batch_size, num_points, num_dims)
        optimizer.zero_grad()
        outputs = model(X_batch) # PointNet forward handles flattening
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(loader.dataset)

# Evaluation PointNet
def test_pointnet(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            # X_batch from loader is (batch_size, num_points, num_dims)
            outputs = model(X_batch) # PointNet forward handles flattening
            _, preds = torch.max(outputs, 1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return correct / total, all_preds, all_labels

# Loop d'entraînement PointNet
epochs = 20
print("\n--- Training PointNet ---")
for epoch in range(epochs):
    # Use the DataLoader with 3D data
    loss = train_pointnet(pointnet, train_loader_pt, criterion, optimizer, device)
    acc, _, _ = test_pointnet(pointnet, test_loader_pt, device)
    print(f"PointNet Epoch {epoch+1}/{epochs} - Loss: {loss:.4f} - Accuracy: {acc:.4f}")

# Final evaluation PointNet with metrics
final_acc_pt, final_preds_pt, final_labels_pt = test_pointnet(pointnet, test_loader_pt, device)

# Get unique labels present in final_labels and filter target_names accordingly
unique_test_labels_pt = np.unique(final_labels_pt)
filtered_target_names_pt = [classes[i] for i in unique_test_labels_pt]

print("\nPointNet Final Metrics:")
print(f"Accuracy: {final_acc_pt:.4f}")
print("Classification Report:")
print(classification_report(final_labels_pt, final_preds_pt, labels=unique_test_labels_pt, target_names=filtered_target_names_pt))
print("Confusion Matrix:")
cm_pt = confusion_matrix(final_labels_pt, final_preds_pt, labels=unique_test_labels_pt)
print(cm_pt)


--- Training PointNet ---
PointNet Epoch 1/20 - Loss: 7.6067 - Accuracy: 0.3276
PointNet Epoch 2/20 - Loss: 4.1219 - Accuracy: 0.3534
PointNet Epoch 3/20 - Loss: 2.5154 - Accuracy: 0.4483
PointNet Epoch 4/20 - Loss: 1.7998 - Accuracy: 0.5431
PointNet Epoch 5/20 - Loss: 1.3293 - Accuracy: 0.4655
PointNet Epoch 6/20 - Loss: 1.3256 - Accuracy: 0.4914
PointNet Epoch 7/20 - Loss: 1.0641 - Accuracy: 0.4741
PointNet Epoch 8/20 - Loss: 0.8270 - Accuracy: 0.4569
PointNet Epoch 9/20 - Loss: 0.7185 - Accuracy: 0.3793
PointNet Epoch 10/20 - Loss: 0.5665 - Accuracy: 0.5000
PointNet Epoch 11/20 - Loss: 0.4540 - Accuracy: 0.4914
PointNet Epoch 12/20 - Loss: 0.3291 - Accuracy: 0.5000
PointNet Epoch 13/20 - Loss: 0.2573 - Accuracy: 0.5086
PointNet Epoch 14/20 - Loss: 0.2277 - Accuracy: 0.4741
PointNet Epoch 15/20 - Loss: 0.2409 - Accuracy: 0.4569
PointNet Epoch 16/20 - Loss: 0.1949 - Accuracy: 0.4914
PointNet Epoch 17/20 - Loss: 0.1623 - Accuracy: 0.4914
PointNet Epoch 18/20 - Loss: 0.1510 - Accuracy:

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# DGCNN Model and helper functions (unchanged from original)
def knn(x, k):
    # x: (batch_size, num_dims, num_points)
    inner = -2 * torch.matmul(x.transpose(2,1), x)
    xx = torch.sum(x**2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2,1)
    idx = pairwise_distance.topk(k=k, dim=-1)[1]   # indices des k plus proches
    return idx

In [ ]:
def get_graph_feature(x, k=20):
    batch_size, num_dims, num_points = x.size()
    idx = knn(x, k)  # (batch_size, num_points, k)

    # Ensure idx_base is on the same device as idx and x
    idx_base = torch.arange(0, batch_size, device=x.device).view(-1,1,1)*num_points
    idx = idx + idx_base
    idx = idx.view(-1)
    x = x.transpose(2,1).contiguous()  # (batch_size, num_points, num_dims)
    feature = x.view(batch_size*num_points, -1)[idx, :]
    feature = feature.view(batch_size, num_points, k, num_dims)
    x = x.view(batch_size, num_points, 1, num_dims).repeat(1,1,k,1)

    feature = torch.cat((feature - x, x), dim=3).permute(0,3,1,2).contiguous()
    return feature  # (batch_size, 2*num_dims, num_points, k)

In [ ]:
class DGCNN(nn.Module):
    def __init__(self, num_classes, k=20):
        super(DGCNN, self).__init__()
        self.k = k
        self.bn1 = nn.BatchNorm2d(64)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)
        self.bn4 = nn.BatchNorm2d(256)

        self.conv1 = nn.Sequential(nn.Conv2d(6, 64, kernel_size=1, bias=False), self.bn1, nn.LeakyReLU(negative_slope=0.2))
        self.conv2 = nn.Sequential(nn.Conv2d(128, 64, kernel_size=1, bias=False), self.bn2, nn.LeakyReLU(negative_slope=0.2))
        self.conv3 = nn.Sequential(nn.Conv2d(128, 128, kernel_size=1, bias=False), self.bn3, nn.LeakyReLU(negative_slope=0.2))
        self.conv4 = nn.Sequential(nn.Conv2d(256, 256, kernel_size=1, bias=False), self.bn4, nn.LeakyReLU(negative_slope=0.2))

        self.linear1 = nn.Linear(512, 256, bias=False)
        self.bn5 = nn.BatchNorm1d(256)
        self.dp1 = nn.Dropout(p=0.5)
        self.linear2 = nn.Linear(256, 128)
        self.bn6 = nn.BatchNorm1d(128)
        self.dp2 = nn.Dropout(p=0.5)
        self.linear3 = nn.Linear(128, num_classes)

    def forward(self, x):
        # Input x is (batch_size, num_dims, num_points)
        batch_size = x.size(0)

        x1 = self.conv1(get_graph_feature(x, k=self.k))  # (B, 64, N, k)
        x1 = x1.max(dim=-1, keepdim=False)[0]             # (B, 64, N)

        x2 = self.conv2(get_graph_feature(x1, k=self.k))
        x2 = x2.max(dim=-1, keepdim=False)[0]

        x3 = self.conv3(get_graph_feature(x2, k=self.k))
        x3 = x3.max(dim=-1, keepdim=False)[0]

        x4 = self.conv4(get_graph_feature(x3, k=self.k))
        x4 = x4.max(dim=-1, keepdim=False)[0]

        x_cat = torch.cat((x1, x2, x3, x4), dim=1)      # (B, 512, N)

        x_global = F.adaptive_max_pool1d(x_cat, 1).view(batch_size, -1)  # (B, 512)
        x_global = self.dp1(F.relu(self.bn5(self.linear1(x_global))))
        x_global = self.dp2(F.relu(self.bn6(self.linear2(x_global))))
        x_global = self.linear3(x_global)
        return x_global


dgcnn = DGCNN(num_classes).to(device)
optimizer = optim.Adam(dgcnn.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Entraînement DGCNN
def train_dgcnn(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        # X_batch from loader is (batch_size, num_points, num_dims)
        # DGCNN expects (batch_size, num_dims, num_points) so we need to transpose
        X_batch = X_batch.transpose(1, 2)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * X_batch.size(0)
    return total_loss / len(loader.dataset)

# Évaluation DGCNN
def test_dgcnn(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            # X_batch from loader is (batch_size, num_points, num_dims)
            # DGCNN expects (batch_size, num_dims, num_points)
            X_batch = X_batch.transpose(1, 2)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    return correct / total, all_preds, all_labels

# Loop d'entraînement DGCNN
print("\n--- Training DGCNN ---")
for epoch in range(epochs):
    # Use the DataLoader with 3D data
    loss = train_dgcnn(dgcnn, train_loader_pt, criterion, optimizer, device)
    acc, _, _ = test_dgcnn(dgcnn, test_loader_pt, device)
    print(f"DGCNN Epoch {epoch+1}/{epochs} - Loss: {loss:.4f} - Accuracy: {acc:.4f}")

# Final evaluation DGCNN with metrics
final_acc_dgcnn, final_preds_dgcnn, final_labels_dgcnn = test_dgcnn(dgcnn, test_loader_pt, device)

# Get unique labels present in final_labels and filter target_names accordingly
unique_test_labels_dgcnn = np.unique(final_labels_dgcnn)
filtered_target_names_dgcnn = [classes[i] for i in unique_test_labels_dgcnn]

print("\nDGCNN Final Metrics:")
print(f"Accuracy: {final_acc_dgcnn:.4f}")
print("Classification Report:")
print(classification_report(final_labels_dgcnn, final_preds_dgcnn, labels=unique_test_labels_dgcnn, target_names=filtered_target_names_dgcnn))
print("Confusion Matrix:")
cm_dgcnn = confusion_matrix(final_labels_dgcnn, final_preds_dgcnn, labels=unique_test_labels_dgcnn)
print(cm_dgcnn)




--- Training DGCNN ---
DGCNN Epoch 1/20 - Loss: 1.6831 - Accuracy: 0.2759
DGCNN Epoch 2/20 - Loss: 1.3124 - Accuracy: 0.4914
DGCNN Epoch 3/20 - Loss: 1.2195 - Accuracy: 0.5603
DGCNN Epoch 4/20 - Loss: 1.1065 - Accuracy: 0.6121
DGCNN Epoch 5/20 - Loss: 1.0422 - Accuracy: 0.6121
DGCNN Epoch 6/20 - Loss: 1.0619 - Accuracy: 0.5776
DGCNN Epoch 7/20 - Loss: 1.0029 - Accuracy: 0.6897
DGCNN Epoch 8/20 - Loss: 0.9490 - Accuracy: 0.6466
DGCNN Epoch 9/20 - Loss: 0.9533 - Accuracy: 0.6293
DGCNN Epoch 10/20 - Loss: 0.9033 - Accuracy: 0.7069
DGCNN Epoch 11/20 - Loss: 0.8734 - Accuracy: 0.6207
DGCNN Epoch 12/20 - Loss: 0.8551 - Accuracy: 0.6293
DGCNN Epoch 13/20 - Loss: 0.8055 - Accuracy: 0.6293
DGCNN Epoch 14/20 - Loss: 0.7799 - Accuracy: 0.6724
DGCNN Epoch 15/20 - Loss: 0.7215 - Accuracy: 0.6638
DGCNN Epoch 16/20 - Loss: 0.6560 - Accuracy: 0.6724
DGCNN Epoch 17/20 - Loss: 0.7095 - Accuracy: 0.5603
DGCNN Epoch 18/20 - Loss: 0.6564 - Accuracy: 0.6724
DGCNN Epoch 19/20 - Loss: 0.6781 - Accuracy: 0.67

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# DGCNN Feature Extraction part
class DGCNN_FeatureExtractor(DGCNN):
    def forward(self, x):
        # Input x is (batch_size, num_dims, num_points)
        batch_size = x.size(0)

        x1 = self.conv1(get_graph_feature(x, k=self.k)).max(dim=-1, keepdim=False)[0]
        x2 = self.conv2(get_graph_feature(x1, k=self.k)).max(dim=-1, keepdim=False)[0]
        x3 = self.conv3(get_graph_feature(x2, k=self.k)).max(dim=-1, keepdim=False)[0]
        x4 = self.conv4(get_graph_feature(x3, k=self.k)).max(dim=-1, keepdim=False)[0]

        x_cat = torch.cat((x1, x2, x3, x4), dim=1)
        x_global = F.adaptive_max_pool1d(x_cat, 1).view(batch_size, -1)
        x_global = F.relu(self.bn5(self.linear1(x_global)))
        x_global = F.relu(self.bn6(self.linear2(x_global)))
        # Ne pas appliquer dropout ni couche finale (classification)
        return x_global

In [ ]:

dgcnn_feat_ext = DGCNN_FeatureExtractor(num_classes).to(device)
# Use the state_dict from the trained DGCNN model
dgcnn_feat_ext.load_state_dict(dgcnn.state_dict())

dgcnn_feat_ext.eval()
all_features = []
all_labels_feat = [] # Use a different variable name to avoid confusion

# Create a DataLoader for the *full* dataset using the 3D data to extract features for all samples
full_dataset_for_features = PointCloudDataset(X, y) # Use X (3D data) here
full_dataloader_for_features = DataLoader(full_dataset_for_features, batch_size=32)


with torch.no_grad():
    for X_batch, y_batch in full_dataloader_for_features:
        X_batch = X_batch.to(device)
        # Transpose for DGCNN feature extractor as it expects (batch_size, num_dims, num_points)
        X_batch = X_batch.transpose(1, 2)
        feats = dgcnn_feat_ext(X_batch)
        all_features.append(feats.cpu().numpy())
        all_labels_feat.append(y_batch.numpy()) # Append to the new variable

X_features = np.concatenate(all_features, axis=0)
y_labels_feat = np.concatenate(all_labels_feat, axis=0) # Concatenate the new variable

# Use the extracted features and labels, splitting them again for consistent evaluation
# Note: This split should ideally match the previous split (X_train, X_test) if you want to evaluate on the *same* test set samples.
# However, splitting the extracted features directly is common practice.
# Use y_labels_feat for stratify
X_train_feat, X_test_feat, y_train_feat, y_test_feat = train_test_split(X_features, y_labels_feat, test_size=0.2, random_state=42, stratify=y_labels_feat) # Add stratify=y_labels_feat

print("\n--- Training Sklearn models on DGCNN features ---")
models_feat = { # Use a different variable name to avoid confusion
    "SVM": SVC(),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Neural Net": MLPClassifier(hidden_layer_sizes=(100,))

}

for name, model in models_feat.items():
    model.fit(X_train_feat, y_train_feat) # Use feature split data
    y_pred_feat = model.predict(X_test_feat) # Use feature split data
    acc_feat = accuracy_score(y_test_feat, y_pred_feat) # Use feature split data

    # Get unique labels present in y_test_feat and filter target_names accordingly
    unique_test_labels_feat = np.unique(y_test_feat)
    filtered_target_names_feat = [classes[i] for i in unique_test_labels_feat]

    print(f"\n{name} on DGCNN features Metrics:")
    print(f"Accuracy: {acc_feat:.4f}")
    print("Classification Report:")
    # Pass unique_test_labels_feat to the labels parameter
    print(classification_report(y_test_feat, y_pred_feat, labels=unique_test_labels_feat, target_names=filtered_target_names_feat))
    print("Confusion Matrix:")
    # The confusion matrix also benefits from knowing the actual labels present
    cm_feat = confusion_matrix(y_test_feat, y_pred_feat, labels=unique_test_labels_feat)
    print(cm_feat)




--- Training Sklearn models on DGCNN features ---

SVM on DGCNN features Metrics:
Accuracy: 0.7500
Classification Report:
              precision    recall  f1-score   support

       Buche       0.75      0.56      0.64        16
   Douglasie       0.85      0.89      0.87        37
       Eiche       0.17      0.25      0.20         4
       Esche       0.00      0.00      0.00         2
      Fichte       0.87      0.84      0.86        32
      Kiefer       0.50      0.60      0.55         5
    Roteiche       0.64      0.70      0.67        20

    accuracy                           0.75       116
   macro avg       0.54      0.55      0.54       116
weighted avg       0.75      0.75      0.75       116

Confusion Matrix:
[[ 9  1  1  0  1  0  4]
 [ 0 33  0  0  2  0  2]
 [ 0  0  1  0  0  1  2]
 [ 0  0  0  0  0  2  0]
 [ 1  2  2  0 27  0  0]
 [ 1  0  0  0  1  3  0]
 [ 1  3  2  0  0  0 14]]

KNN on DGCNN features Metrics:
Accuracy: 0.7672
Classification Report:
              precisi

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Neural Net on DGCNN features Metrics:
Accuracy: 0.7328
Classification Report:
              precision    recall  f1-score   support

       Buche       0.80      0.50      0.62        16
   Douglasie       0.78      0.86      0.82        37
       Eiche       0.25      0.50      0.33         4
       Esche       0.33      0.50      0.40         2
      Fichte       0.81      0.78      0.79        32
      Kiefer       0.75      0.60      0.67         5
    Roteiche       0.74      0.70      0.72        20

    accuracy                           0.73       116
   macro avg       0.64      0.64      0.62       116
weighted avg       0.76      0.73      0.74       116

Confusion Matrix:
[[ 8  2  1  1  2  0  2]
 [ 0 32  0  0  3  0  2]
 [ 0  1  2  0  0  0  1]
 [ 0  0  0  1  0  1  0]
 [ 1  3  3  0 25  0  0]
 [ 1  0  0  0  1  3  0]
 [ 0  3  2  1  0  0 14]]


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
